# 02 — County Triage Clustering

## Approach: KMeans Unsupervised Clustering

There is no column in FEMA administrative data labelled "needs fast-track" or
"high priority". Supervised learning is therefore not applicable — we have no
ground truth labels to train on.

Instead we use **KMeans clustering** to let the data naturally group counties
into vulnerability tiers. We then map those clusters to business labels
(Priority 1 / 2 / 3) based on the cluster centroids — the cluster with the
highest composite vulnerability score becomes Priority 1 (Immediate Escalation).

## Features used

| Feature | Source | Why |
|---|---|---|
| `svi_score` | CDC SVI | Structural vulnerability: poverty, housing, minorities, language barriers |
| `pop_density` | CDC SVI (computed) | Scale of exposure — same disaster hits more people in dense areas |
| `prior_disasters_5yr` | FEMA PA history | Cumulative burden — counties with repeated declarations have less recovery capacity |

All three are scaled with StandardScaler before clustering so no single feature
dominates due to its unit of measurement.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

PROCESSED = '../data/processed/'

df = pd.read_csv(PROCESSED + 'county_disaster_matrix.csv', low_memory=False)
print(f'Loaded: {df.shape}')
print(df[['svi_score','pop_density','prior_disasters_5yr']].describe().round(3))

In [ ]:
# Keep rows with all three features present
FEATURES = ['svi_score', 'pop_density', 'prior_disasters_5yr']
df_cluster = df.dropna(subset=FEATURES).copy()
df_cluster['prior_disasters_5yr'] = df_cluster['prior_disasters_5yr'].fillna(0)

# Cap pop_density at 99th percentile to reduce outlier influence
p99 = df_cluster['pop_density'].quantile(0.99)
df_cluster['pop_density'] = df_cluster['pop_density'].clip(upper=p99)

print(f'Rows available for clustering: {len(df_cluster):,}')
print(f'Pop density capped at 99th pct: {p99:.1f} people/sqmi')

X = df_cluster[FEATURES].values
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(f'\nScaled feature matrix shape: {X_scaled.shape}')

## Step 1 — Validate k=3 with Elbow Method + Silhouette Score

We test k=2 through k=8 to confirm that k=3 is the right choice.
- **Elbow method**: plot inertia (within-cluster sum of squares) — look for the "elbow"
- **Silhouette score**: measures how well-separated clusters are (higher = better, max=1)

We expect k=3 to be justified because:
1. FEMA operationally works with 3-tier triage systems
2. The data has three natural groupings: high-vulnerability dense, moderate, low-vulnerability

In [ ]:
K_RANGE = range(2, 9)
inertias    = []
silhouettes = []

for k in K_RANGE:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(list(K_RANGE), inertias, 'bo-', linewidth=2)
ax1.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='k=3 (chosen)')
ax1.set_xlabel('Number of clusters (k)')
ax1.set_ylabel('Inertia (within-cluster SS)')
ax1.set_title('Elbow Method')
ax1.legend()

ax2.plot(list(K_RANGE), silhouettes, 'go-', linewidth=2)
ax2.axvline(x=3, color='red', linestyle='--', alpha=0.7, label='k=3 (chosen)')
ax2.set_xlabel('Number of clusters (k)')
ax2.set_ylabel('Silhouette Score')
ax2.set_title('Silhouette Score by k')
ax2.legend()

plt.suptitle('Cluster Validation — Elbow + Silhouette', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED + 'elbow_silhouette.png', dpi=150)
plt.show()

print(f'\nSilhouette scores:')
for k, s in zip(K_RANGE, silhouettes):
    marker = ' ← chosen' if k == 3 else ''
    print(f'  k={k}: {s:.4f}{marker}')

In [ ]:
# Fit KMeans with k=3
km = KMeans(n_clusters=3, random_state=42, n_init=10)
df_cluster['cluster'] = km.fit_predict(X_scaled)

# Centroids in original (unscaled) space for interpretation
centroids = scaler.inverse_transform(km.cluster_centers_)
centroid_df = pd.DataFrame(centroids, columns=FEATURES)
centroid_df.index.name = 'cluster'
print('Cluster centroids (original scale):')
print(centroid_df.round(4))

## Step 2 — Data-Driven Cluster Labeling

We do NOT hardcode which cluster number maps to which priority.
KMeans assigns cluster IDs (0, 1, 2) arbitrarily — they may differ across runs.

Instead we rank clusters by a **composite vulnerability score**:
```
vulnerability_score = svi_centroid × 0.5 + norm(pop_density_centroid) × 0.3 + norm(prior_disasters_centroid) × 0.2
```
The cluster with the highest score → Priority 1 (Immediate Escalation)
The cluster with the lowest score → Priority 3 (Standard Review)

This ensures the labeling is always correct regardless of cluster ID assignment.

In [ ]:
# Compute composite vulnerability score per cluster centroid
# Normalise each feature to 0-1 range across the 3 centroids
def minmax(arr):
    rng = arr.max() - arr.min()
    return (arr - arr.min()) / rng if rng > 0 else arr * 0

svi_norm    = minmax(centroid_df['svi_score'].values)
dens_norm   = minmax(centroid_df['pop_density'].values)
prior_norm  = minmax(centroid_df['prior_disasters_5yr'].values)

centroid_df['vulnerability_score'] = svi_norm * 0.5 + dens_norm * 0.3 + prior_norm * 0.2

# Rank: highest score = Priority 1
rank = centroid_df['vulnerability_score'].rank(ascending=False).astype(int)
cluster_to_priority = rank.to_dict()

PRIORITY_LABELS = {1: 'Priority 1 — Immediate Escalation',
                   2: 'Priority 2 — Targeted Support',
                   3: 'Priority 3 — Standard Review'}

PRIORITY_COLORS = {1: '#e74c3c', 2: '#f39c12', 3: '#2ecc71'}

df_cluster['priority']       = df_cluster['cluster'].map(cluster_to_priority)
df_cluster['priority_label'] = df_cluster['priority'].map(PRIORITY_LABELS)

print('Cluster → Priority mapping:')
for c, p in sorted(cluster_to_priority.items()):
    print(f'  Cluster {c} → Priority {p}: {PRIORITY_LABELS[p]}')
    print(f'    SVI={centroid_df.loc[c,"svi_score"]:.3f}  '
          f'Density={centroid_df.loc[c,"pop_density"]:.1f}  '
          f'Prior={centroid_df.loc[c,"prior_disasters_5yr"]:.2f}  '
          f'Score={centroid_df.loc[c,"vulnerability_score"]:.3f}')

print(f'\nPriority distribution:')
print(df_cluster['priority_label'].value_counts().sort_index())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SVI vs Pop Density
for p in [1, 2, 3]:
    mask = df_cluster['priority'] == p
    axes[0].scatter(df_cluster.loc[mask, 'svi_score'],
                    df_cluster.loc[mask, 'pop_density'],
                    c=PRIORITY_COLORS[p], label=PRIORITY_LABELS[p],
                    alpha=0.4, s=15)
axes[0].set_xlabel('SVI Score (0=least vulnerable, 1=most vulnerable)')
axes[0].set_ylabel('Population Density (people/sqmi, capped at p99)')
axes[0].set_title('County Clusters: SVI vs Population Density')
axes[0].legend(fontsize=8)

# SVI vs Prior Disasters
for p in [1, 2, 3]:
    mask = df_cluster['priority'] == p
    axes[1].scatter(df_cluster.loc[mask, 'svi_score'],
                    df_cluster.loc[mask, 'prior_disasters_5yr'],
                    c=PRIORITY_COLORS[p], label=PRIORITY_LABELS[p],
                    alpha=0.4, s=15)
axes[1].set_xlabel('SVI Score')
axes[1].set_ylabel('Prior Disasters in 5yr Window')
axes[1].set_title('County Clusters: SVI vs Prior Disaster Exposure')
axes[1].legend(fontsize=8)

plt.suptitle('County Triage Engine — Cluster Assignments', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(PROCESSED + 'cluster_scatter.png', dpi=150)
plt.show()

In [ ]:
# Merge priority labels back to full matrix
output = df.merge(
    df_cluster[['fips','disasterNumber','cluster','priority','priority_label']],
    on=['fips','disasterNumber'], how='left'
)

output.to_csv(PROCESSED + 'county_triage_scored.csv', index=False)

print(f'Saved: county_triage_scored.csv ({len(output):,} rows)')
print(f'\nPriority coverage: {output["priority"].notna().mean():.1%} of county-disaster rows scored')
print(output['priority_label'].value_counts())